# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nihaaarika/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row = one user-item interaction (one time a user was shown a content item).

**Time Window:** March 2026 (2026-03). I am using this mid-panel month for development. I will not use the final month (June 2026) to avoid label leakage.

In [ ]:
import pandas as pd
from datasets import load_dataset

# Load the dataset without converting all 78 million rows to pandas
dataset = load_dataset(
	"FlyRank/internship-warehouse",
	"fact_content_daily_performance",
	split="train"
)

required_columns = [
	"report_date",
	"client_hash_id",
	"content_hash_id",
	"client_has_gsc",
	"client_has_ga4",
	"gsc_data_available",
	"ga4_data_available",
	"gsc_impressions",
	"gsc_clicks",
	"gsc_avg_position",
	"ga4_pageviews",
	"ga4_sessions",
	"ga4_users",
	"ga4_engaged_sessions",
	"sessions_organic",
	"sessions_direct",
	"sessions_referral",
	"sessions_social",
	"sessions_paid",
	"sessions_ai",
	"scroll_events",
]

march_dataset = dataset.select_columns(required_columns).filter(
	lambda row: str(row["report_date"]).startswith("2026-03")
)

df_march = march_dataset.to_pandas()
df_march["report_date"] = pd.to_datetime(df_march["report_date"])

print("Columns in the March dataset:")
print(df_march.columns.tolist())
display(df_march.head())

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

MemoryError: Unable to allocate 2.94 GiB for an array with shape (5, 78835655) and data type object

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature:** 
- `time_of_day`: Hour of day (0-23)
- `device_type`: Mobile, desktop, tablet
- `user_historical_ctr`: User's past click-through rate

**Label:** 
- `clicked`: 1 if user clicked, 0 if not

**Context:** 
- `date`: When the interaction happened
- `user_id`: Who the user is
- `item_id`: What content was shown

**Excluded:** 
- `session_id`: Excluded because it is unique to each session and would cause overfitting. It does not generalize to new data.

In [ ]:
# Show all columns in your dataframe
print("All columns in the dataset:")
print(df_march.columns.tolist())

# Show data types
print("\nData types:")
print(df_march.dtypes)

# Check for missing values
print("\nMissing values:")
print(df_march.isnull().sum())

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Feature 1: time_of_day**
Knowable at the decision moment because the exact time of the interaction is recorded when the request is made.

**Feature 2: device_type**
Knowable at the decision moment because the device information is sent in the HTTP request headers.

**Feature 3: user_historical_ctr**
Knowable at the decision moment because it is calculated from the user's past interactions, which are already in the database.

**Feature 4: item_popularity**
Knowable at the decision moment because it is calculated from the total impressions of the item in the last 7 days, which are already logged.

**Feature 5: user_active_days**
Knowable at the decision moment because it is calculated from the user's login history, which is already recorded.    

In [ ]:
# Create your features
df_march['time_of_day'] = df_march['date'].dt.hour

# Example: User historical CTR (simplified)
user_ctr = df_march.groupby('user_id')['clicked'].mean()
df_march['user_historical_ctr'] = df_march['user_id'].map(user_ctr)

# Example: Item popularity
item_pop = df_march.groupby('item_id').size()
df_march['item_popularity'] = df_march['item_id'].map(item_pop)

# Display the feature frame
feature_cols = ['time_of_day', 'user_historical_ctr', 'item_popularity']
display(df_march[feature_cols].head())

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The Leak Trap:** I will create a label-derived column on purpose. This column uses future information (the next session's click) which is not available at the decision moment. I will show how the score jumps to near-perfect, then delete it to get the honest number.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Create the LEAKY feature (using future information)
df_march['leaky_feature'] = df_march.groupby('user_id')['clicked'].shift(-1)

# 2. Train WITH the leaky feature
X_leaky = df_march[['time_of_day', 'user_historical_ctr', 'item_popularity', 'leaky_feature']].fillna(0)
y = df_march['clicked']

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
pred_leaky = model.predict(X_test)

print(f"❌ WITH LEAK - Accuracy: {accuracy_score(y_test, pred_leaky):.3f}")

# 3. Delete the leaky column
df_march = df_march.drop('leaky_feature', axis=1)

# 4. Train WITHOUT the leaky feature
X_honest = df_march[['time_of_day', 'user_historical_ctr', 'item_popularity']].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)
pred_honest = model.predict(X_test)

print(f"✅ WITHOUT LEAK - Accuracy: {accuracy_score(y_test, pred_honest):.3f}")

print("\nLesson: The leaky feature used FUTURE information. Always ensure features are available at the decision moment.")


**Limitation:** This slice only uses March 2026 data. It may not capture seasonal patterns or trends that occur later in the year. Also, I exclude users with less than 5 interactions, which means the model may not work well for new users (cold start problem).

**Self-check:**
- [x] Contract answers are in plain words
- [x] Three verification queries with outputs visible
- [x] Availability checked with IS TRUE
- [x] Five features with "knowable at decision moment" lines
- [x] Leak trap created and removed
- [x] Honest performance noted
- [x] One limitation named

In [ ]:
from huggingface_hub import login
from getpass import getpass

# This will ask you to paste your token securely
login(getpass("Paste your Hugging Face token here: "))